In [ ]:
from __future__ import annotations

import json
import joblib
import sys
from pathlib import Path

repo_root = Path(__file__).resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from data622.config import (
    PROCESSED_DATA_DIR,
    MODELS_DIR,
    MODEL_FEATURES,
    TARGET_COL,
    TARGET_LOG_COL,
)
from data622.dataset import (
    load_salary_data,
    filter_model_population,
    add_tenure_proxy,
    split_by_year,
)
from data622.features import add_feature_columns, make_preprocessor


def main():
    print("Loading salary data...")
    df = load_salary_data()

    print(f"Raw rows: {len(df):,}")

    print("Filtering modeling population...")
    df = filter_model_population(df)

    print("Adding tenure proxy...")
    df = add_tenure_proxy(df)

    print("Adding engineered features...")
    df = add_feature_columns(df)

    print(f"Final modeling rows: {len(df):,}")

    train_df, valid_df, test_df = split_by_year(df)

    print(f"Train rows: {len(train_df):,}")
    print(f"Valid rows: {len(valid_df):,}")
    print(f"Test rows:  {len(test_df):,}")

    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    train_path = PROCESSED_DATA_DIR / "salary_model_train.parquet"
    valid_path = PROCESSED_DATA_DIR / "salary_model_valid.parquet"
    test_path = PROCESSED_DATA_DIR / "salary_model_test.parquet"
    full_path = PROCESSED_DATA_DIR / "salary_model_full.parquet"

    train_df.to_parquet(train_path, index=False)
    valid_df.to_parquet(valid_path, index=False)
    test_df.to_parquet(test_path, index=False)
    df.to_parquet(full_path, index=False)

    print("Saved split datasets.")

    feature_cols = [c for c in MODEL_FEATURES if c in train_df.columns]
    preprocessor = make_preprocessor(train_df)
    preprocessor.fit(train_df[feature_cols])

    preprocessor_path = MODELS_DIR / "salary_preprocessor.joblib"
    joblib.dump(preprocessor, preprocessor_path)
    print(f"Saved preprocessor to {preprocessor_path}")

    metadata = {
        "target_column": TARGET_COL,
        "target_log_column": TARGET_LOG_COL,
        "feature_columns": feature_cols,
        "train_shape": train_df.shape,
        "valid_shape": valid_df.shape,
        "test_shape": test_df.shape,
    }

    metadata_path = PROCESSED_DATA_DIR / "salary_model_metadata.json"
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved metadata to {metadata_path}")


if __name__ == "__main__":
    main()